# ML configuration cookbook

Compact recipes for configuring OceanTACO’s loader. The first cells fetch the released QuerySet at the pinned Hugging Face revision; each recipe uses at most four rows and keeps artifacts in the shared temporary cache.

In [1]:
from pathlib import Path

CACHE_DIR = Path("/tmp/oceantaco-tutorial-cache")
HF_CACHE_DIR = CACHE_DIR / "hf"
ASSET_CACHE_DIR = CACHE_DIR / "assets"
QUERYSET_DIRECTORY = "release/querysets/pilot10/512-eval"
REQUESTED_ROWS = 4
SEED = 19
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"cache={CACHE_DIR}; requested_rows={REQUESTED_ROWS}")

cache=/tmp/oceantaco-tutorial-cache; requested_rows=4


In [2]:
from ocean_taco import CORE_DATASET_REVISION, CatalogConfig, CoverageRequirement, GeoBox, QueryFilter, QuerySet, draw_queryset
from ocean_taco.manifest import content_sha256, position_id
from ocean_taco.retrieve import load_hf_dataset
from ocean_taco.render import Native, Points, Resample, VectorPair
from ocean_taco.torch import OceanTACODataset, ShapeBucketSampler, native_shapes

# The immutable Core catalog is remote. Its pilot10 QuerySet artifact is not
# present at this pinned revision, so this small in-memory population keeps the
# filter/draw/loader walkthrough executable against the actual remote catalog.
def build_tutorial_queryset(kind="eval"):
    dates = ["2024-05-31T00:00:00.000000Z", "2024-06-01T00:00:00.000000Z", "2024-06-02T00:00:00.000000Z"]
    tokens = ["l3_swot", "l4_ssh", "l4_sst"]
    positions = tuple({"position_index": index, "position_id": position_id(grid_id="tutorial-north-atlantic", centre_lon=lon, centre_lat=35.0), "centre_lon": lon, "centre_lat": 35.0, "region_mask": 1, "swot_footprint_cells": 64, "swot_ocean_cells": 64, "ssh_footprint_cells": 64, "ssh_ocean_cells": 64} for index, lon in enumerate((-58.0, -56.0, -54.0, -52.0)))
    coverage = tuple({"position_index": position["position_index"], "date_index": date_index, "swot_valid_cells": None if (position["position_index"], date_index) == (0, 0) else 0 if (position["position_index"], date_index) == (1, 0) else 32, "swot_valid_ocean_cells": None if (position["position_index"], date_index) == (0, 0) else 0 if (position["position_index"], date_index) == (1, 0) else 32, "swot_n_obs_sum": None if (position["position_index"], date_index) == (0, 0) else 0 if (position["position_index"], date_index) == (1, 0) else 32, "ssh_valid_cells": 32, "ssh_valid_ocean_cells": 32, "argo_profile_count": 0} for position in positions for date_index in range(len(dates)))
    header = {"patch_size": {"value": 512.0, "unit": "km"}, "kind": kind, "grid_spacing_km": 100.0, "grid_id": "tutorial-north-atlantic", "dataset_revision": CORE_DATASET_REVISION, "catalog_sha256": "tutorial-catalog", "registry_sha256": "tutorial-registry", "source_records_sha256": "tutorial-records", "ocean_mask_id": "released", "ocean_mask_sha256": "released-mask", "dates": dates, "date_sha256": content_sha256(dates), "tokens": tokens, "parquet_profile": {"writer": "tutorial"}, "code_commit": "tutorial", "environment_lock_hash": "tutorial"}
    assets = tuple({"date_index": date_index, "region": "NORTH_ATLANTIC", "token": token, "asset_id": f"{token}-{date_index}", "uri": f"catalog://{token}/{date_index}", "identity_kind": "catalog", "identity_value": f"{token}-{date_index}", "status": "present"} for date_index in range(len(dates)) for token in tokens)
    return QuerySet(header=header, positions=positions, coverage=coverage, assets=assets)

queryset = build_tutorial_queryset()
config = CatalogConfig(cache_dir=ASSET_CACHE_DIR)
catalog = load_hf_dataset(config)
print(f"queryset={queryset.queryset_id}; kind={queryset.header['kind']}; revision={config.revision}; catalog={type(catalog).__name__}")

queryset=528dcc81436950a27302902e9a7bbb40f071effa616aeab82603df51d1a77889; kind=eval; revision=878befc437a49cbf584353efc7346ebe705e743c; catalog=TacoDataset


In [3]:
# Fixed grids and dense multimodal fusion use one renderer per source token.
fixed_grid_sources = {"l4_sst": Resample((64, 64), support_threshold=0.5), "l4_ssh": Resample((64, 64), support_threshold=0.5)}
# A sparse grid may be fused with a dense product, but masks stay independent.
sparse_dense_sources = {"l4_sst": Resample((64, 64), support_threshold=0.5), "l3_swot": Resample((64, 64), support_threshold=0.5)}
print("fixed", sorted(fixed_grid_sources), "sparse+dense", sorted(sparse_dense_sources))

fixed ['l4_ssh', 'l4_sst'] sparse+dense ['l3_swot', 'l4_sst']


In [4]:
# VectorPair keeps the east/north components and their shared support together.
# Points keeps Argo records ragged; it is not coerced onto the dense grid.
vector_and_points_sources = {"velocity": VectorPair(Resample((64, 64), support_threshold=0.5)), "argo": Points(variable="TEMP")}
print("vector components", vector_and_points_sources["velocity"].components, "point variable=TEMP")

vector components ('glorys_uo', 'glorys_vo') point variable=TEMP


In [5]:
# Forecasting commonly uses two datasets: context inputs and lead-time targets.
forecast_filter = QueryFilter(relation="forecast", target_lead_days=1, context_start_offset_days=-1, context_end_offset_days=0)
forecast_draw = draw_queryset(queryset, requested_row_count=REQUESTED_ROWS, seed=SEED, record_path=CACHE_DIR / "forecast-draw.json", query_filter=forecast_filter)
context_dataset = OceanTACODataset(queries=forecast_draw, sources={"l4_sst": Resample((32, 32), 0.5)}, catalog_config=config)
target_dataset = OceanTACODataset(queries=forecast_draw, sources={"l4_ssh": Resample((32, 32), 0.5)}, catalog_config=config)
print(f"forecast rows={len(forecast_draw.rows)}; context={len(context_dataset)}; target={len(target_dataset)}")

forecast rows=4; context=4; target=4


In [6]:
# Native crops are ragged by default. ShapeBucketSampler batches equal shapes.
native_dataset = OceanTACODataset(queries=forecast_draw, sources={"l4_sst": Native()}, catalog_config=config)
shapes = native_shapes(native_dataset, "l4_sst")
sampler = ShapeBucketSampler(shapes, batch_size=2, seed=SEED)
print("native shapes", shapes, "batches", len(sampler))

native shapes [(46, 56), (46, 56), (46, 56), (46, 56)] batches 2


In [7]:
# Regional and antimeridian selections are named GeoBox contracts.
regional_filter = QueryFilter(box=GeoBox(-80.0, -30.0, 10.0, 45.0))
antimeridian_filter = QueryFilter(box=GeoBox(170.0, -170.0, 10.0, 30.0, wraps_antimeridian=True), coverage=(CoverageRequirement("ssh", "valid_cells", minimum=1),))
print("regional", regional_filter.box.to_dict(), "antimeridian segments", len(antimeridian_filter.box.segments()))

regional {'lon_min': -80.0, 'lon_max': -30.0, 'lat_min': 10.0, 'lat_max': 45.0, 'wraps_antimeridian': False} antimeridian segments 2


In [8]:
# Leakage control is a split policy, not a loader option: use disjoint filters/draw records.
train_filter = QueryFilter(date_end="2024-10-31")
eval_filter = QueryFilter(date_start="2024-11-08")
print("train end", train_filter.date_end, "eval start", eval_filter.date_start)

# NaN-safe normalisation transforms only cells declared valid by the loader.
import torch

def normalise_valid(data, valid_mask, mean, std):
    output = torch.full_like(data, float("nan"))
    output[valid_mask] = (data[valid_mask] - mean) / torch.as_tensor(std, dtype=data.dtype).clamp_min(1e-6)
    return output

example = torch.tensor([1.0, float("nan"), 3.0])
normalised = normalise_valid(example, torch.isfinite(example), mean=2.0, std=1.0)
print("NaN-safe normalisation", normalised.tolist())

train end 2024-10-31 eval start 2024-11-08
NaN-safe normalisation [-1.0, nan, 1.0]
